# EDA: Distribution Check for Single Family Residential Sales

This notebook explores the distributions of `ClosePrice`, `LivingArea`, `Bedrooms`, `Bathrooms`, and `LotSize`.

The task document says to restrict the analysis to:

- `PropertyType = Residential`
- `PropertySubType = SingleFamilyResidence`

For `LotSize`, I use `LotSizeSquareFeet` because it is a numeric lot size field.


In [ ]:
import pandas as pd
import numpy as np

pd.options.display.float_format = '{:,.2f}'.format


## 1. Load the merged data

I only read the columns needed for this EDA, so the notebook runs faster.


In [ ]:
use_columns = [
    'PropertyType',
    'PropertySubType',
    'ClosePrice',
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet'
]

data = pd.read_csv('../data/merged_crmls_sold.csv', usecols=use_columns, low_memory=False)

data.head()


In [ ]:
print('Rows before filtering:', len(data))
print('Columns:', data.shape[1])


## 2. Filter to the required property type

The analysis below only uses Residential and SingleFamilyResidence rows.


In [ ]:
single_family = data[
    (data['PropertyType'] == 'Residential') &
    (data['PropertySubType'] == 'SingleFamilyResidence')
].copy()

print('Rows after filtering:', len(single_family))
single_family.head()


## 3. Clean the numeric columns

The assignment asks for Bedrooms, Bathrooms, and LotSize. In the dataset, I use:

- `BedroomsTotal` as Bedrooms
- `BathroomsTotalInteger` as Bathrooms
- `LotSizeSquareFeet` as LotSize


In [ ]:
numeric_columns = [
    'ClosePrice',
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet'
]

for column in numeric_columns:
    single_family[column] = pd.to_numeric(single_family[column], errors='coerce')

single_family = single_family.rename(columns={
    'BedroomsTotal': 'Bedrooms',
    'BathroomsTotalInteger': 'Bathrooms',
    'LotSizeSquareFeet': 'LotSize'
})

eda_columns = ['ClosePrice', 'LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize']

single_family[eda_columns].head()


## 4. Missing values

Before looking at distributions, I check how many missing values each variable has.


In [ ]:
missing_values = single_family[eda_columns].isna().sum()
missing_percent = single_family[eda_columns].isna().mean() * 100

missing_table = pd.DataFrame({
    'missing_count': missing_values,
    'missing_percent': missing_percent
})

missing_table


## 5. Summary statistics

This table gives the basic distribution summary for each variable.


In [ ]:
summary_table = single_family[eda_columns].describe().T
summary_table


## 6. Quantiles

Quantiles make it easier to see the typical range and high-end outliers.


In [ ]:
quantile_table = single_family[eda_columns].quantile([
    0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99
]).T

quantile_table


## 7. Skewness

Positive skewness means the distribution has a long right tail.


In [ ]:
skew_table = single_family[eda_columns].skew().to_frame(name='skewness')
skew_table


## 8. Histogram-style frequency tables

These tables are similar to histograms. They count how many rows fall into each value range.


In [ ]:
for column in eda_columns:
    print()
    print(column)
    clean_values = single_family[column].dropna()
    bins = pd.cut(clean_values, bins=10)
    print(bins.value_counts().sort_index())


## 9. Quantile bins

The equal-width bins above can be affected by extreme outliers. Quantile bins are another way to inspect the distribution.


In [ ]:
for column in eda_columns:
    print()
    print(column)
    clean_values = single_family[column].dropna()
    bins = pd.qcut(clean_values, q=10, duplicates='drop')
    print(bins.value_counts().sort_index())


## 10. Simple outlier check

I use the IQR rule to count possible low and high outliers.


In [ ]:
outlier_rows = []

for column in eda_columns:
    clean_values = single_family[column].dropna()

    q1 = clean_values.quantile(0.25)
    q3 = clean_values.quantile(0.75)
    iqr = q3 - q1

    low_limit = q1 - 1.5 * iqr
    high_limit = q3 + 1.5 * iqr

    low_outliers = (clean_values < low_limit).sum()
    high_outliers = (clean_values > high_limit).sum()

    outlier_rows.append({
        'variable': column,
        'low_limit': low_limit,
        'high_limit': high_limit,
        'low_outliers': low_outliers,
        'high_outliers': high_outliers
    })

outlier_table = pd.DataFrame(outlier_rows)
outlier_table


## Optional histograms

If `matplotlib` is installed, this cell will draw histograms. If not, the notebook still works using the frequency tables above.


In [ ]:
try:
    import matplotlib.pyplot as plt

    single_family[eda_columns].hist(bins=50, figsize=(14, 10))
    plt.suptitle('Raw Distributions', y=1.02)
    plt.tight_layout()
    plt.show()

    log_data = single_family[eda_columns].copy()

    for column in eda_columns:
        log_data[column] = log_data[column].where(log_data[column] >= 0)
        log_data[column] = np.log1p(log_data[column])

    log_data.hist(bins=50, figsize=(14, 10))
    plt.suptitle('Log Distributions using log1p', y=1.02)
    plt.tight_layout()
    plt.show()

except ModuleNotFoundError:
    print('matplotlib is not installed, so I used the frequency tables above instead.')


## 11. Quick observations

- `ClosePrice`, `LivingArea`, and `LotSize` are right-skewed.
- `Bedrooms` and `Bathrooms` are count-like variables, so their distributions are more discrete.
- The quantile table and outlier table show large high-end values, especially for `ClosePrice` and `LotSize`.
- For modeling later, it may be useful to consider log transformations for very skewed variables such as `ClosePrice`, `LivingArea`, and `LotSize`.
